# Benchmark: data products vs on-demand queries

Requires gold `integrated_taxi_trips` and data products from `run_gold`.
Shared SQL lives in `src/queries/analytical.py`; helpers in `src/benchmark/`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, ensure_runtime

ensure_runtime()

from src.lake import GOLD, read_delta
from src.benchmark.evaluate import evaluate_query
from src.queries.analytical import (
    QUERY_1_MONTHLY_ZONE_DEMAND as query_1,
    QUERY_2_WEATHER_DISTANCE as query_2,
    QUERY_3_PM25_DEMAND as query_3,
    QUERY_4_ZONE_WEATHER_SENSITIVITY as query_4,
    QUERY_5_PEAK_HOURS_BY_DOW as query_5,
    QUERY_6_MONTHLY_TRENDS as query_6,
)

spark = create_spark("benchmark")
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView(
    "integrated_taxi_trips"
)

:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a8509ad5-da37-45d6-993b-c9f43b8b5b76;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 105ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

### Query 1–4: on-demand vs data products

In [2]:
prod = GOLD / "data_products"

evaluate_query(
    spark,
    "Query 1: Monthly Taxi Demand per Zone",
    spark.sql(query_1),
    read_delta(spark, prod / "taxi_zone_monthly_demand").select(
        "trip_month", "pickup_location_id", "pickup_borough", "pickup_zone",
        "total_trips", "active_days", "avg_daily_trips",
    ),
    prod / "taxi_zone_monthly_demand",
)

evaluate_query(
    spark,
    "Query 2: Average Distance by Weather",
    spark.sql(query_2),
    read_delta(spark, prod / "weather_impact_summary").select(
        "temp_category", "wind_category", "trip_count", "avg_distance_miles"
    ),
    prod / "weather_impact_summary",
)

evaluate_query(
    spark,
    "Query 3: Air Quality vs Demand",
    spark.sql(query_3),
    read_delta(spark, prod / "air_quality_demand_summary").select(
        "pm25_level", "trips", "observed_hours", "trips_per_hour"
    ),
    prod / "air_quality_demand_summary",
)

evaluate_query(
    spark,
    "Query 4: Zone Weather Demand Variation",
    spark.sql(query_4),
    read_delta(spark, prod / "zone_weather_sensitivity").select(
        "pickup_zone", "coldest", "cool", "warm", "warmest", "pct_variation"
    ),
    prod / "zone_weather_sensitivity",
)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/lake/gold/data_products/taxi_zone_monthly_demand.

### Query 5–6: AQE on/off

In [ ]:
evaluate_query(
    spark,
    "Query 5: Peak Travel Hours by Day of Week",
    aqe_query=query_5,
)
evaluate_query(
    spark,
    "Query 6: Monthly Trends in Taxi Demand",
    aqe_query=query_6,
)
spark.stop()